# DATA-2 — génération de mfn_0000..0019 (commit 23875ab, graine 20261021), renommage, recouvrement, index

In [ ]:
import importlib, os, subprocess, sys, time, datetime, faulthandler
def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')
os.makedirs('/kaggle/working', exist_ok=True)
_TRACE = open('/kaggle/working/data2_cell1_trace.txt', 'a')
def _say(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line, flush=True)
    _TRACE.write(line + '\n'); _TRACE.flush()
_say('data2 install start', _utc())
REQUIRED_OUTSIDE_IMAGE = ["Pillow>=12.3,<13", "fonttools>=4.60,<5", "jsonschema>=4.25,<5", "lxml>=6,<7", "numpy>=2.3,<3", "shapely>=2.1,<3"]
for pkg in REQUIRED_OUTSIDE_IMAGE:
    _say('pip start', pkg, _utc())
    _t0 = time.monotonic()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--timeout', '30', '--retries', '1', pkg],
                   check=True, timeout=900)
    _say('pip end', pkg, _utc(), round(time.monotonic() - _t0, 1), 's')
faulthandler.dump_traceback_later(600, exit=True, file=_TRACE)
for mod in ["numpy", "PIL", "lxml", "fontTools", "jsonschema", "shapely"]:
    _say('import module', mod, _utc())
    importlib.import_module(mod)
_say('imports ok', _utc())
faulthandler.cancel_dump_traceback_later()


In [ ]:
import datetime, hashlib, importlib, importlib.util, json, os, platform, shutil, subprocess, sys, zipfile
from pathlib import Path

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

def _imp(name):
    print('import', name, _utc(), flush=True)
    return importlib.import_module(name)

print('data2 cell 2 start', _utc(), flush=True)
KIT_CANDIDATES = ["/kaggle/input/babaorum-data2-kit/scripts/data2_generate.py", "/kaggle/input/datasets/mabaku/babaorum-data2-kit/scripts/data2_generate.py"]
OUT = Path("/kaggle/working")
RAW = OUT / "gen_raw"
SRC = OUT / "heritage"
BUNDLE = OUT / "bundle"
GEN_OUT = OUT / "gen_out"
FRAG_OUT = OUT / "frag_out"
STATE = {}

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

E24_SHA_FULL = "714bc571db6a4e394027ec4e98cd4c997bde5ff54c735fbc8f32d9cdc4218654"
BUNDLE_V5_SHA = "ca41c09272248c5096f2547fe738eee622db5af50414182537ac8e94e43367f5"
OVERLAP_ACCEPT = {}          # §14 (1)/(2): only OVERLAP_TEXT_FLAGGED, never OVERLAP_PAGE
OVERLAP_MENTION = ""
def check_reference(path):
    """The E2E-4 v3 report: whole sha256 and alphabet size 74 are checked before any use."""
    assert sha256(path) == E24_SHA_FULL, "E2E-4 report: sha256 differs from the frozen value"
    assert json.loads(Path(path).read_text())["alphabet_size"] == 74, "E2E-4 report: alphabet size is not 74"

_hits = [Path(c) for c in KIT_CANDIDATES if Path(c).is_file()]        # explicit candidates only, no listing
assert len(_hits) == 1, ("kit found %d times among the candidates" % len(_hits), [str(h) for h in _hits])
KIT_DIR = _hits[0].parent.parent
def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod
C = _load("data2_common", KIT_DIR / "scripts" / "data2_common.py")     # copied LD-0 functions (provenance in header)
print("kit", KIT_DIR, _utc(), flush=True)

def d_controls():
    for m in ["numpy", "PIL", "lxml", "fontTools", "jsonschema", "shapely"]:
        _imp(m)
    print("versions", {m: getattr(sys.modules[m], "__version__", None) for m in ["numpy", "PIL", "lxml", "fontTools", "jsonschema", "shapely"]}, flush=True)

def d_discover():
    assert not (OUT / "data2_generation_manifest.json").exists(), "the output must start empty"
    found = C.discover_inputs(Path("/kaggle/input"), relpaths={"bundle": "stage_s_bundle_v5.zip.bin", "kit": "scripts/data2_generate.py", "reference": "e2e4_extraction_report.json", "run3_mf_dir": "e2e3/mf"})
    STATE["found"] = found
    print("discovery roots (names only):", found["roots_used"], flush=True)
    assert sha256(found["bundle"]) == BUNDLE_V5_SHA, "bundle v5 sha differs"
    check_reference(found["reference"])

def d_bundle_clean():
    shutil.rmtree(BUNDLE, ignore_errors=True)
    BUNDLE.mkdir(parents=True)
    shutil.rmtree(RAW, ignore_errors=True)
    shutil.rmtree(SRC, ignore_errors=True)

def d_mille_check():
    manifest = json.loads((BUNDLE / "bundle_manifest.json").read_text())
    for rel, want in manifest["mille_feuilles_sha256"].items():
        got = sha256(SRC / rel[len("heritage-synth/"):])
        assert got == want, ("mille-feuilles file differs", rel)
    print("mille-feuilles: 51 files checked", flush=True)

def d_cleanup(rec):
    mpath = OUT / "data2_notebook_manifest.json"
    mani = json.loads(mpath.read_text()) if mpath.exists() else {}
    mani["exit_status"] = rec.get("status")
    mani["durations_s"] = rec.get("durations_s")
    mani["cleanup"] = {"raw_removed": str(RAW), "heritage_clone_removed": str(SRC), "bundle_removed": str(BUNDLE)}
    mpath.write_text(json.dumps(mani, indent=2))
    for d in (RAW, SRC, BUNDLE):
        shutil.rmtree(d, ignore_errors=True)

SMOKE_PAGES = 0
SMOKE_SEED = 20261021
STEPS = [
    {"name": "controls", "fn": d_controls},
    {"name": "discover", "fn": d_discover},
    {"name": "bundle_clean", "fn": d_bundle_clean},
    {"name": "unzip", "argv": lambda: [sys.executable, "-c",
                                       "import sys, zipfile; zipfile.ZipFile(sys.argv[1]).extractall(sys.argv[2])",
                                       str(STATE["found"]["bundle"]), str(BUNDLE)], "timeout_s": 900},
    {"name": "clone", "argv": lambda: ["git", "clone", "-q", str(BUNDLE / "heritage-synth.gitbundle"), str(SRC)],
     "timeout_s": 600},
    {"name": "checkout", "argv": lambda: ["git", "-C", str(SRC), "checkout", "-q", "23875ab05425ce2aa07742bf314d02436c2f5642"],
     "timeout_s": 600},
    {"name": "mille_check", "fn": d_mille_check},
    {"name": "generate", "argv": lambda: [sys.executable, str(KIT_DIR / "scripts" / "data2_generate.py"),
                                          "generate", "--src", str(SRC), "--raw", str(RAW)] +
     (["--pages", str(SMOKE_PAGES), "--seed", str(SMOKE_SEED)] if SMOKE_PAGES else []),
     "cwd": str(KIT_DIR), "timeout_s": 2700},
    {"name": "rename", "argv": lambda: [sys.executable, str(KIT_DIR / "scripts" / "data2_generate.py"),
                                        "rename", "--raw", str(RAW), "--out", str(GEN_OUT),
                                        "--pages", str(SMOKE_PAGES or 20)],
     "cwd": str(KIT_DIR), "timeout_s": 600},
    {"name": "overlap", "argv": lambda: [sys.executable, str(KIT_DIR / "scripts" / "data2_generate.py"),
                                         "overlap", "--out", str(GEN_OUT), "--mf-dir", str(STATE["found"]["run3_mf_dir"]),
                                         "--reference-report", str(STATE["found"]["reference"]),
                                         "--pages", str(SMOKE_PAGES or 20)],
     "cwd": str(KIT_DIR), "timeout_s": 600},
    {"name": "index", "argv": lambda: [sys.executable, str(KIT_DIR / "scripts" / "data2_generate.py"),
                                       "index", "--out", str(GEN_OUT), "--pages", str(SMOKE_PAGES or 20)],
     "cwd": str(KIT_DIR), "timeout_s": 300},
]

EXIT = C.execute_pipeline(OUT, STEPS, heartbeat_interval_s=60.0, cleanup=d_cleanup)
print("EXIT", json.dumps({k: EXIT[k] for k in ("status", "step", "exit_code")}), flush=True)
